In [2]:
# stdlib
import base64
import json

# third party
import torch as th

# syft absolute
import syft as sy
from syft.core.plan.plan_builder import ROOT_CLIENT
from syft.core.plan.plan_builder import PLAN_BUILDER_VM
from syft.core.plan.plan_builder import make_plan
from syft.core.plan.translation.torchscript.plan_translate import (
    translate as translate_to_ts,
)
from syft.federated.model_centric_fl_client import ModelCentricFLClient
from syft.lib.python.int import Int
from syft.lib.python.list import List

In [3]:
class MLP(sy.Module):
    """
    Simple model with method for loss and hand-written backprop.
    """

    def __init__(self, torch_ref) -> None:
        super(MLP, self).__init__(torch_ref=torch_ref)
        self.fc1 = torch_ref.nn.Linear(784, 100)
        self.relu = torch_ref.nn.ReLU()
        self.fc2 = torch_ref.nn.Linear(100, 10)


    def forward(self, x):
        self.z1 = self.fc1(x)
        self.a1 = self.relu(self.z1)
        return self.fc2(self.a1)

    def backward(self, X, error):
        z1_grad = (error @ self.fc2.state_dict()["weight"]) * (self.a1 > 0).float()
        fc1_weight_grad = z1_grad.t() @ X
        fc1_bias_grad = z1_grad.sum(0)
        fc2_weight_grad = error.t() @ self.a1
        fc2_bias_grad = error.sum(0)
        return fc1_weight_grad, fc1_bias_grad, fc2_weight_grad, fc2_bias_grad

    def softmax_cross_entropy_with_logits(self, logits, target, batch_size):
        probs = self.torch_ref.softmax(logits, dim=1)
        loss = -(target * self.torch_ref.log(probs)).sum(dim=1).mean()
        loss_grad = (probs - target) / batch_size
        return loss, loss_grad

    def accuracy(self, logits, targets, batch_size):
        pred = self.torch_ref.argmax(logits, dim=1)
        targets_idx = self.torch_ref.argmax(targets, dim=1)
        acc = pred.eq(targets_idx).sum().float() / batch_size
        return acc

In [4]:
def set_remote_model_params(module_ptrs, params_list_ptr):
    """Sets the model parameters into traced model"""
    param_idx = 0
    for module_name, module_ptr in module_ptrs.items():
        for param_name, _ in PLAN_BUILDER_VM.store[
            module_ptr.id_at_location
        ].data.named_parameters():
            module_ptr.register_parameter(param_name, params_list_ptr[param_idx])
            param_idx += 1

# Create the model
local_model = MLP(th)

# Dummy inputs
bs = 3
classes_num = 10
model_params_zeros = sy.lib.python.List(
    [th.nn.Parameter(th.zeros_like(param)) for param in local_model.parameters()]
)

@make_plan
def training_plan(
    xs=th.randn(bs, 28 * 28),
    ys=th.nn.functional.one_hot(th.randint(0, classes_num, [bs]), classes_num),
    batch_size=th.tensor([bs]),
    lr=th.tensor([0.1]),
    params=model_params_zeros,
):
    # send the model to plan builder (but not its default params)
    # this is required to build the model inside the Plan
    model = local_model.send(ROOT_CLIENT, send_parameters=False)

    # set model params from input
    set_remote_model_params(model.modules, params)

    # forward
    logits = model(xs)

    # loss
    loss, loss_grad = model.softmax_cross_entropy_with_logits(
        logits, ys, batch_size
    )

    # backward
    grads = model.backward(xs, loss_grad)

    # SGD step
    updated_params = tuple(
        param - lr * grad for param, grad in zip(model.parameters(), grads)
    )

    # accuracy
    acc = model.accuracy(logits, ys, batch_size)

    # return things
    return (loss, acc, *updated_params)

In [5]:
ts_plan = translate_to_ts(training_plan)

# Let's examine its contents
print(ts_plan.torchscript.code)

/opt/homebrew/anaconda3/envs/py39/lib/python3.9/site-packages/torch/tensor.py:587: RuntimeWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  warnings.warn('Iterating over a tensor might cause the trace to be incorrect. '
/opt/homebrew/anaconda3/envs/py39/lib/python3.9/site-packages/syft/lib/torch/tensor_util.py:53: TracerWarning: Converting a tensor to a NumPy array might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  numpy_tensor = tensor.detach().numpy()
/opt/homebrew/anaconda3/envs/py39/lib/python3.9/site-packages/syft/lib/torch/tensor_util.py:105: TracerWarning: torch.from_numpy results are registered as constants in the trace. You can safely ignore this warni

def forward(self,
    input: Tensor,
    tensor: Tensor,
    tensor0: Tensor,
    tensor1: Tensor,
    argument_5: List[Tensor]) -> List[Tensor]:
  weight, tensor2, tensor3, tensor4, = argument_5
  input0 = torch.addmm(tensor2, input, torch.t(weight), beta=1, alpha=1)
  input1 = torch.relu(input0)
  _0 = torch.addmm(tensor4, input1, torch.t(tensor3), beta=1, alpha=1)
  tensor5 = torch.softmax(_0, 1, None)
  tensor6 = torch.mul(tensor, torch.log(tensor5))
  tensor7 = torch.sum(tensor6, [1], False, dtype=None)
  _1 = torch.neg(torch.mean(tensor7, dtype=None))
  tensor8 = torch.sub(tensor5, tensor, alpha=1)
  tensor9 = torch.div(tensor8, tensor0)
  tensor10 = torch.detach(tensor3)
  tensor11 = torch.matmul(tensor9, tensor10)
  tensor12 = torch.gt(input1, 0)
  _2 = torch.to(tensor12, 6, False, False, None)
  tensor13 = torch.mul(tensor11, _2)
  tensor14 = torch.t(tensor13)
  _3 = torch.matmul(tensor14, input)
  _4 = torch.sum(tensor13, [0], False, dtype=None)
  tensor15 = torch.t(tensor9)


/opt/homebrew/anaconda3/envs/py39/lib/python3.9/site-packages/torch/jit/_trace.py:934: TracerWarning: Encountering a list at the output of the tracer might cause the trace to be incorrect, this is only valid if the container structure does not change based on the module's inputs. Consider using a constant container instead (e.g. for `list`, use a `tuple` instead. for `dict`, use a `NamedTuple` instead). If you absolutely need this and know the side effects, pass strict=False to trace() to allow this behavior.
  module._c._create_method_from_trace(


In [6]:
ts_plan.torchscript.save("syft.pt")

In [7]:
training_plan.parameters()

AttributeError: 'Plan' object has no attribute 'parameters'

In [9]:
list(ts_plan.torchscript.parameters())

[]

In [10]:
print(training_plan)

Plan
Allowed executions:	not defined
Remaining executions:	not defined
Inputs:
		xs:	TensorPointer
		ys:	TensorPointer
		batch_size:	TensorPointer
		lr:	TensorPointer
		params:	ListPointer
Actions:
		93 Actions
Outputs:
		TensorPointer
		TensorPointer
		AnyPointer
		AnyPointer
		AnyPointer
		AnyPointer

Plan code:
"""
@make_plan
def training_plan(
    xs=th.randn(bs, 28 * 28),
    ys=th.nn.functional.one_hot(th.randint(0, classes_num, [bs]), classes_num),
    batch_size=th.tensor([bs]),
    lr=th.tensor([0.1]),
    params=model_params_zeros,
):
    # send the model to plan builder (but not its default params)
    # this is required to build the model inside the Plan
    model = local_model.send(ROOT_CLIENT, send_parameters=False)

    # set model params from input
    set_remote_model_params(model.modules, params)

    # forward
    logits = model(xs)

    # loss
    loss, loss_grad = model.softmax_cross_entropy_with_logits(
        logits, ys, batch_size
    )

    # backward
    grad

In [11]:
kwarg_names = list(training_plan.inputs.keys())
print(kwarg_names)

['xs', 'ys', 'batch_size', 'lr', 'params']


In [12]:
args = tuple(
        PLAN_BUILDER_VM.store[training_plan.inputs[name].id_at_location].data
        for name in kwarg_names)
print(args)

(tensor([[-0.8984, -0.9593,  0.8855,  ...,  0.2430,  0.6454,  0.0778],
        [ 0.1560,  0.0738,  0.0643,  ...,  0.3488, -0.8082, -1.9684],
        [ 0.9323, -1.2205, -1.2569,  ...,  2.1613, -0.2866,  0.4638]]), tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]]), tensor([3]), tensor([0.1000]), [Parameter containing:
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], requires_grad=True), Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
  

In [26]:
from typing import Any
from typing import Dict as TypeDict
from typing import List as TypeList

class PlanWrapper(th.nn.Module):
        """
        Plan needs to be executed with kwargs while torchscript needs to be executed with args,
        so we need to have kwarg names inside torchscript to pass them into Plan.
        We use nn.Module to store input kwarg names inside torchscript.
        """

        def __init__(self, kwarg_names: TypeList[str]):
            _kwarg_names: TypeList[str]
            super(PlanWrapper, self).__init__()
            self._kwarg_names = kwarg_names

        def forward(self, *args: Any) -> Any:
            kwarg_ptrs: TypeDict[str, Pointer] = {}

            # Since Syft Plan needs pointers as args,
            # reverse-map actual arg values to pointers
            # by looking up these args in VM store
            for name, arg in zip(self._kwarg_names, args):
                ptr = get_pointer_to_data_in_store(
                    PLAN_BUILDER_VM.store, arg, ROOT_CLIENT
                )
                if not ptr:
                    traceback_and_raise(f"Could not map '{name}' arg value to Pointer")
                kwarg_ptrs[name] = ptr

            # Execute Plan in the same VM where it was built!
            res = training_plan(PLAN_BUILDER_VM, PLAN_BUILDER_VM.verify_key, **kwarg_ptrs)  # type: ignore
            return res

In [27]:
from syft.lib.python.primitive_interface import PyPrimitive
from syft.core.plan.translation.torchscript.plan_translate import get_pointer_to_data_in_store
from syft.core.plan import Plan

args = tuple(arg.upcast() if isinstance(arg, PyPrimitive) else arg for arg in args)
wrapper = PlanWrapper(kwarg_names)
ts = th.jit.trace(wrapper, args, check_trace=False)

[2022-03-03T15:04:14.434211+0100][CRITICAL][logger]][74293] <class 'syft.core.store.store_memory.MemoryStore'> __getitem__ error <UID: c3864fa01fa74183853e0c0f2e9d723f> <UID: c3864fa01fa74183853e0c0f2e9d723f>
[2022-03-03T15:04:14.434941+0100][CRITICAL][logger]][74293] <UID: c3864fa01fa74183853e0c0f2e9d723f>


KeyError: <UID: c3864fa01fa74183853e0c0f2e9d723f>